In [ ]:
# IMPORTS
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py

RESULTS_DIR = os.path.join(os.getcwd())

def snapshot_path_discrete(run_index):
    base = os.path.join(RESULTS_DIR, f'snapshots{run_index}')
    return os.path.join(base, f'snapshots{run_index}_s1.h5')



In [ ]:
# BULK PROCESSING AND VISUALISATION
Ma_indices = []
Ma_list = []
Ma_f_max = 1e7

index_file = os.path.join(RESULTS_DIR, 'index_record.txt')
with open(index_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            index_str, param_str = line.split()
            if float(param_str) > Ma_f_max:
                continue
            Ma_indices.append(int(index_str))
            Ma_list.append(float(param_str))

print('Indices:', Ma_indices)
print('Ma list:', Ma_list)

fails = []  # run indices to skip, if any
filtered = [(idx, Ma) for idx, Ma in zip(Ma_indices, Ma_list) if idx not in fails]
Ma_indices, Ma_list = zip(*filtered) if filtered else ([], [])
Ma_indices, Ma_list = list(Ma_indices), list(Ma_list)

success_bools = []
final_depths = []
final_times = []

for i in range(len(Ma_list)):
    try:
        print('Processing Ma=' + str(Ma_list[i]))
        filepath = snapshot_path_discrete(Ma_indices[i])
        with h5py.File(filepath, 'r') as f:
            s_data = f['tasks']['zb'][:]
            times = f['scales']['sim_time'][:]
        if s_data[-1] > 0:
            success_bools.append(True)
        else:
            success_bools.append(False)
        if success_bools[-1]:
            final_depths.append(0.0)
        else:
            final_depths.append(float(-s_data[-1, 0, 0, 0]))
        if success_bools[-1]:
            final_times.append(float(np.interp(0, s_data[:, 0, 0, 0], times)))
        else:
            final_times.append(np.inf)
    except Exception as e:
        print(e)
        print(str(Ma_list[i]) + ' failed somewhere')

print('Stored Marangoni numbers:')
print(Ma_list)
print('Successfully reached interface?')
print(success_bools)
print('Depth at termination:')
print(final_depths)
print('Time at interface:')
print(final_times)

fail_list = [f for f, b in zip(Ma_list, success_bools) if not b]
if fail_list:
    print('Largest subcritical Ma:', max(fail_list))



In [ ]:
# Plot a subset of trajectories (edit Ma_subset as needed)
Ma_subset = [70, 75, 80]

Ma_indices = []
Ma_list = []
index_file = os.path.join(RESULTS_DIR, 'index_record.txt')
with open(index_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            index_str, param_str = line.split()
            if float(param_str) in Ma_subset:
                Ma_indices.append(int(index_str))
                Ma_list.append(float(param_str))

print('Indices:', Ma_indices)
print('Ma list:', Ma_list)

for i in range(len(Ma_list)):
    try:
        print('Processing Ma=' + str(Ma_list[i]))
        filepath = snapshot_path_discrete(Ma_indices[i])
        with h5py.File(filepath, 'r') as f:
            s_data = f['tasks']['zb'][:]
            times = f['scales']['sim_time'][:]
            mass_data = f['tasks']['total_mass'][:]
        plt.figure(0)
        plt.plot(times, np.squeeze(mass_data))
        plt.figure(1)
        plt.plot(times, np.squeeze(s_data))
    except Exception as e:
        print(e)

plt.figure(0)
plt.ylim(-2, 2)
plt.figure(1)
plt.ylim(-1, 0)
plt.xlim(0, 1)
plt.savefig('trajectories_DF_a.svg')
plt.legend([str(Ma) for Ma in Ma_subset])

